## 🔧 Fixed: MLflow Configuration Error

### ❌ Original Problem
```
mlflow.legacy_databricks_cli.configure.provider.InvalidConfigurationError: 
You haven't configured the CLI yet!
```

**Cause:** Using `subprocess.run()` in Databricks notebook loses the managed MLflow context.

---

### ✅ Solution: Direct Import

**BEFORE (Cells 2-4):**
```python
result = subprocess.run(['python', 'scripts/run_pipeline.py', ...])
```
❌ Runs in separate shell → loses Databricks authentication

**AFTER (All cells updated):**
```python
from scripts.run_pipeline import main
main(args)
```
✅ Runs in notebook context → preserves MLflow integration

---

## 📊 Updated Workflow

| Cell | Action | Method | Status |
|------|--------|--------|--------|
| **1** | Install packages | %pip | ✅ |
| **2** | Train baseline | Direct import | ✅ Fixed |
| **3** | Fine-tune | Direct import | ✅ Fixed |
| **4** | Export ONNX | Direct import | ✅ Fixed |
| **5** | Test inference | Direct import | ✅ Works |

---

## 🚀 Benefits

✅ **MLflow context preserved** - No authentication errors
✅ **Better error messages** - Stack traces in notebook
✅ **Faster execution** - No subprocess overhead
✅ **Interactive debugging** - Can inspect variables
✅ **Native Databricks** - Works with managed services

---

## 💡 When to Use Each Method

**Direct Import (Current approach):**
- ✅ Databricks notebooks
- ✅ Need MLflow integration
- ✅ Interactive development
- ✅ Better debugging

**Subprocess:**
- ✅ Production cron jobs
- ✅ Shell scripts
- ✅ Process isolation needed
- ✅ Command-line tools

---

## ⚡ Ready to Run!

All cells have been updated. You can now:
1. **Run Cell 2** - Train baseline model
2. **Run Cell 3** - Fine-tune (optional)
3. **Run Cell 4** - Export ONNX
4. **Run Cell 5** - Test inference

In [0]:
%pip install -r requirements.txt

In [0]:
# Run the MLOps training pipeline
import sys
import os

# Change to project directory
project_root = '/Workspace/Users/trannguyentoanphat1592005@gmail.com/mlops-e2e'
os.chdir(project_root)
sys.path.insert(0, project_root)

print(f"Current directory: {os.getcwd()}")
print("\nStarting MLOps pipeline...\n")
print("="*80)

# Import and run training directly (avoid subprocess to preserve MLflow context)
import argparse
from scripts.run_pipeline import main

# Create args object
class Args:
    input = 'data/raw/student_mental_health_burnout_relabeled.csv'
    target = 'burnout_level'
    test_size = 0.2
    experiment = 'Student Burnout Prediction'
    mlflow_uri = None  # Use Databricks managed MLflow

args = Args()

try:
    # Run training
    main(args)
    print("\n" + "="*80)
    print("✅ Pipeline completed successfully!")
    print("="*80)
except Exception as e:
    print("\n" + "="*80)
    print(f"❌ Pipeline failed: {e}")
    print("="*80)
    import traceback
    traceback.print_exc()

In [0]:
# Restart Python to reload updated scripts
dbutils.library.restartPython()

In [0]:
# Fine-tune a trained LightGBM model from MLflow
import sys
import os

project_root = '/Workspace/Users/trannguyentoanphat1592005@gmail.com/mlops-e2e'
os.chdir(project_root)
# Make sure project_root is in sys.path BEFORE importing scripts
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Fine-tuning LightGBM Model")
print("="*80)

# ======================================================================
# CONFIGURATION: Choose ONE of the following methods
# ======================================================================

# METHOD 1: Use Run ID (UUID) - Works in both Local and Databricks
BASE_RUN_IDENTIFIER = "6ca55f540219470381434c7ae622c89d"  # ← From Cell 3

# METHOD 2: Use Run Name - Works in both Local and Databricks
# BASE_RUN_IDENTIFIER = "202604271449_lightgbm_baseline"  # ← From Cell 3

# METHOD 3: Manual override
# MANUAL_IDENTIFIER = None  # Set this to override (run_id OR run_name)
# if MANUAL_IDENTIFIER:
#     BASE_RUN_IDENTIFIER = MANUAL_IDENTIFIER

print(f"\n✅ Base Model Identifier: {BASE_RUN_IDENTIFIER}")
print("   (Can be run_id OR run_name - both work!)\n")
print("Tuning Configuration:")
print("  - Optuna hyperparameter search: ENABLED")
print("  - Number of trials: 10")
print("\nStarting fine-tuning...\n")
print("-"*80)

# Import and run fine-tuning directly (preserve MLflow context)
from scripts.fine_tune import main

class Args:
    base_run_identifier = BASE_RUN_IDENTIFIER  # Accept both run_id and run_name
    experiment_name = 'Student Burnout Prediction'
    mlflow_uri = None  # Use Databricks managed MLflow
    tune = True  # Enable Optuna hyperparameter search
    n_trials = 10  # Number of Optuna trials

args = Args()

try:
    # Run fine-tuning
    results = main(args)
    
    if results:
        print("\n" + "="*80)
        print("✅ Fine-tuning completed successfully!")
        print("="*80)
        print("\n📌 NEXT STEPS:")
        print("  1. Copy EITHER of these identifiers from output above:")
        print("     - Run ID (UUID): Works everywhere")
        print("     - Run Name (timestamp): Works everywhere")
        print("  2. Paste into Cell 6 (MODEL_RUN_IDENTIFIER) to export")
        print("="*80)
        
        # Store for next cell
        finetuned_run_id = results['run_id']
        finetuned_run_name = results['run_name']
        print(f"\n💾 Saved to variables:")
        print(f"   finetuned_run_id = '{finetuned_run_id}'")
        print(f"   finetuned_run_name = '{finetuned_run_name}'")
except Exception as e:
    print("\n" + "="*80)
    print(f"❌ Fine-tuning failed: {e}")
    print("="*80)
    import traceback
    traceback.print_exc()
    print("\n💡 TIP: If base model not found, check the identifier is correct")
    print("   You can use EITHER run_id OR run_name from MLflow UI.")

In [0]:
# Export trained/fine-tuned LightGBM model to ONNX format
import sys
import os

project_root = '/Workspace/Users/trannguyentoanphat1592005@gmail.com/mlops-e2e'
os.chdir(project_root)
sys.path.insert(0, project_root)

print("Export LightGBM Model to ONNX")
print("="*80)

# ======================================================================
# CONFIGURATION: Choose ONE of the following methods
# ======================================================================

# METHOD 1: Use Run ID (UUID) - Works in both Local and Databricks
MODEL_RUN_IDENTIFIER = "5eacdd4be2bc4ad3aad00ead1fd1535b"  # ← Baseline from Cell 3

# METHOD 2: Use Run Name - Works in both Local and Databricks
# MODEL_RUN_IDENTIFIER = "202604271449_lightgbm_baseline"  # ← From Cell 3

# METHOD 3: Use fine-tuned model (after running Cell 5)
# Uncomment and update after fine-tuning:
# MODEL_RUN_IDENTIFIER = finetuned_run_id  # From Cell 5
# or
# MODEL_RUN_IDENTIFIER = finetuned_run_name  # From Cell 5

# METHOD 4: Use "latest" to auto-find the most recent run
# MODEL_RUN_IDENTIFIER = "latest"

print(f"\n✅ Model Identifier: {MODEL_RUN_IDENTIFIER}")
print("   (Can be run_id, run_name, or 'latest')\n")
print("Export Configuration:")
print("  - Output format: ONNX")
print("  - Benchmark samples: 1000")
print("  - Throughput metric: Will be logged to MLflow")
print("\nStarting ONNX export...\n")
print("-"*80)

# Import export function
from scripts.export_model import main

class Args:
    run_identifier = MODEL_RUN_IDENTIFIER  # Accept run_id, run_name, or "latest"
    experiment_name = 'Student Burnout Prediction'
    output_dir = os.path.join(project_root, 'artifacts')
    mlflow_uri = None  # Use Databricks managed MLflow
    benchmark_samples = 1000

args = Args()

try:
    # Run ONNX export
    result = main(args)
    
    if result:
        print("\n" + "="*80)
        print("✅ ONNX export completed successfully!")
        print("="*80)
        print("\nArtifacts created:")
        print(f"  - {result['onnx_path']}")
        print(f"  - {result['metadata_path']}")
        print("\nMLflow updated:")
        print(f"  - throughput_onnx: {result['throughput']:.0f} samples/sec")
        print("\n🚀 Model ready for serving!")
        print("   Test it using Cell 7 (Test Inference API)")
        print("="*80)
    
except Exception as e:
    print("\n" + "="*80)
    print(f"❌ ONNX export failed: {e}")
    print("="*80)
    import traceback
    traceback.print_exc()
    
    print("\n💡 TROUBLESHOOTING:")
    print("-"*80)
    print("\nPossible causes:")
    print("  1. Model not found (check identifier is correct)")
    print("  2. Model artifact wasn't logged properly")
    print("  3. onnxmltools dependencies missing")
    print("\nYou can use EITHER:")
    print("  - run_id (UUID): e.g., '6ca55f540219470381434c7ae622c89d'")
    print("  - run_name: e.g., '202604271449_lightgbm_baseline'")
    print("  - 'latest': Automatically find most recent run")
    print("="*80)

In [0]:
# Test the inference API with sample student data
import sys
sys.path.append('/Workspace/Users/trannguyentoanphat1592005@gmail.com/mlops-e2e')

from src.serving.inference import predict

print("Testing Inference API with 3 sample profiles...\n")
print("="*80)

# Test Case 1: High Burnout Risk Profile
print("\nTEST 1: High Burnout Risk Profile")
print("-" * 40)
high_risk = {
    'gender': 'Male',
    'age': 22,
    'course': 'BTech',
    'year': '4th',
    'daily_study_hours': 8.0,
    'daily_sleep_hours': 4.0,
    'screen_time_hours': 9.0,
    'stress_level': 'High',
    'anxiety_score': 9,
    'depression_score': 8,
    'academic_pressure_score': 10,
    'financial_stress_score': 8,
    'social_support_score': 3,
    'physical_activity_hours': 0.0,
    'sleep_quality': 'Poor',
    'attendance_percentage': 45.0,
    'cgpa': 5.5,
    'internet_quality': 'Poor'
}

result1 = predict(high_risk)
print(f"Input: Final year student, 8hrs study, 4hrs sleep, high stress")
print(f"Prediction: {result1}")

# Test Case 2: Low Burnout Risk Profile
print("\nTEST 2: Low Burnout Risk Profile")
print("-" * 40)
low_risk = {
    'gender': 'Female',
    'age': 20,
    'course': 'BA',
    'year': '2nd',
    'daily_study_hours': 3.0,
    'daily_sleep_hours': 8.0,
    'screen_time_hours': 3.0,
    'stress_level': 'Low',
    'anxiety_score': 2,
    'depression_score': 1,
    'academic_pressure_score': 3,
    'financial_stress_score': 2,
    'social_support_score': 9,
    'physical_activity_hours': 2.0,
    'sleep_quality': 'Good',
    'attendance_percentage': 95.0,
    'cgpa': 9.2,
    'internet_quality': 'Good'
}

result2 = predict(low_risk)
print(f"Input: Second year student, balanced lifestyle, good support")
print(f"Prediction: {result2}")

# Test Case 3: Medium Burnout Risk Profile
print("\nTEST 3: Medium Burnout Risk Profile")
print("-" * 40)
medium_risk = {
    'gender': 'Male',
    'age': 21,
    'course': 'BTech',
    'year': '3rd',
    'daily_study_hours': 4.0,
    'daily_sleep_hours': 7.0,
    'screen_time_hours': 5.0,
    'stress_level': 'Medium',
    'anxiety_score': 5,
    'depression_score': 5,
    'academic_pressure_score': 6,
    'financial_stress_score': 5,
    'social_support_score': 7,
    'physical_activity_hours': 1.0,
    'sleep_quality': 'Average',
    'attendance_percentage': 85.0,
    'cgpa': 7.5,
    'internet_quality': 'Good'
}

result3 = predict(medium_risk)
print(f"Input: Third year student, moderate stress, average metrics")
print(f"Prediction: {result3}")

print("\n" + "="*80)
print("\n Inference API Test Complete!")
print("\nModel Details:")
print("  - Framework: LightGBM (migrated from XGBoost)")
print("  - Format: ONNX (onnxruntime inference)")
print("  - Features: 23 features (after one-hot encoding)")
print("  - Model Size: 1.38 MB")
print("  - ROC AUC: 0.733")